<a href="https://colab.research.google.com/github/Harshada900/pyspark-practise/blob/main/6_Broadcast_Join_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Session5").config("spark.sql.shuffle.partitions", "10").getOrCreate()

In [3]:
# creating sample data with 9999 rows
emp_data = [(i, f"Employee_{i}", i % 5 + 101)
            for i in range(1, 10000)]
emp_cols = ["emp_id", "name", "dept_id"]
emp_df = spark.createDataFrame(emp_data, emp_cols)

In [4]:

# Small department lookup table
dept_data = [(101, "Engineering"),
             (102, "Marketing"),
             (103, "HR"),
             (104, "Finance"),
             (105, "Sales")]
dept_cols = ["dept_id", "dept_name"]
dept_df = spark.createDataFrame(dept_data, dept_cols)

In [6]:
emp_df.show(5)

+------+----------+-------+
|emp_id|      name|dept_id|
+------+----------+-------+
|     1|Employee_1|    102|
|     2|Employee_2|    103|
|     3|Employee_3|    104|
|     4|Employee_4|    105|
|     5|Employee_5|    101|
+------+----------+-------+
only showing top 5 rows


### **Task: Join them using broadcast join**

In [7]:
import pyspark.sql.functions as f

emp_df.join(f.broadcast(dept_df), on="dept_id", how ="left").show()   # speeds up joining data

+-------+------+-----------+-----------+
|dept_id|emp_id|       name|  dept_name|
+-------+------+-----------+-----------+
|    102|     1| Employee_1|  Marketing|
|    103|     2| Employee_2|         HR|
|    104|     3| Employee_3|    Finance|
|    105|     4| Employee_4|      Sales|
|    101|     5| Employee_5|Engineering|
|    102|     6| Employee_6|  Marketing|
|    103|     7| Employee_7|         HR|
|    104|     8| Employee_8|    Finance|
|    105|     9| Employee_9|      Sales|
|    101|    10|Employee_10|Engineering|
|    102|    11|Employee_11|  Marketing|
|    103|    12|Employee_12|         HR|
|    104|    13|Employee_13|    Finance|
|    105|    14|Employee_14|      Sales|
|    101|    15|Employee_15|Engineering|
|    102|    16|Employee_16|  Marketing|
|    103|    17|Employee_17|         HR|
|    104|    18|Employee_18|    Finance|
|    105|    19|Employee_19|      Sales|
|    101|    20|Employee_20|Engineering|
+-------+------+-----------+-----------+
only showing top

In [8]:
emp_df.join(dept_df, on="dept_id", how ="left").show()

+-------+------+-----------+-----------+
|dept_id|emp_id|       name|  dept_name|
+-------+------+-----------+-----------+
|    105|     4| Employee_4|      Sales|
|    105|     9| Employee_9|      Sales|
|    105|    14|Employee_14|      Sales|
|    105|    19|Employee_19|      Sales|
|    104|     3| Employee_3|    Finance|
|    104|     8| Employee_8|    Finance|
|    104|    13|Employee_13|    Finance|
|    104|    18|Employee_18|    Finance|
|    102|     1| Employee_1|  Marketing|
|    102|     6| Employee_6|  Marketing|
|    102|    11|Employee_11|  Marketing|
|    102|    16|Employee_16|  Marketing|
|    102|    21|Employee_21|  Marketing|
|    103|     2| Employee_2|         HR|
|    101|     5| Employee_5|Engineering|
|    103|     7| Employee_7|         HR|
|    101|    10|Employee_10|Engineering|
|    103|    12|Employee_12|         HR|
|    101|    15|Employee_15|Engineering|
|    103|    17|Employee_17|         HR|
+-------+------+-----------+-----------+
only showing top

Then check the execution plan to confirm BroadcastHashJoin

In [15]:
emp_df.join(f.broadcast(dept_df), on="dept_id", how ="left").explain()    #Look for BroadcastHashJoin in the output — that confirms it worked.

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [dept_id#2L, emp_id#0L, name#1, dept_name#4]
   +- BroadcastHashJoin [dept_id#2L], [dept_id#3L], LeftOuter, BuildRight, false
      :- InMemoryTableScan [emp_id#0L, name#1, dept_id#2L]
      :     +- InMemoryRelation [emp_id#0L, name#1, dept_id#2L], StorageLevel(disk, memory, deserialized, 1 replicas)
      :           +- *(1) Scan ExistingRDD[emp_id#0L,name#1,dept_id#2L]
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=359]
         +- Filter isnotnull(dept_id#3L)
            +- Scan ExistingRDD[dept_id#3L,dept_name#4]




Check Table Size Before Broadcasting


In [13]:

emp_df.count()

9999

In [14]:
emp_df.cache()

DataFrame[emp_id: bigint, name: string, dept_id: bigint]

# Check current threshold (default is 10MB)

In [16]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

'10485760b'

# Change it, tell Spark to auto broadcast tables under 50MB

In [17]:
spark.conf.set("saprk.sql.autoBroadcastJoinThreshold", "50m") # or u can write 50 * 1024 *1024

In [18]:
# Disable auto broadcast completely
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [19]:
emp_df.join(dept_df, on="dept_id", how ="left").show()    # with broadcast it was 1s now it shows 2s after disabling it !

+-------+------+-------------+-----------+
|dept_id|emp_id|         name|  dept_name|
+-------+------+-------------+-----------+
|    101|     5|   Employee_5|Engineering|
|    101|    10|  Employee_10|Engineering|
|    101|    15|  Employee_15|Engineering|
|    101|    20|  Employee_20|Engineering|
|    101|  5125|Employee_5125|Engineering|
|    101|  5130|Employee_5130|Engineering|
|    101|  5135|Employee_5135|Engineering|
|    101|  5140|Employee_5140|Engineering|
|    102|     1|   Employee_1|  Marketing|
|    102|     6|   Employee_6|  Marketing|
|    102|    11|  Employee_11|  Marketing|
|    102|    16|  Employee_16|  Marketing|
|    102|    21|  Employee_21|  Marketing|
|    102|  5121|Employee_5121|  Marketing|
|    102|  5126|Employee_5126|  Marketing|
|    102|  5131|Employee_5131|  Marketing|
|    102|  5136|Employee_5136|  Marketing|
|    102|  5141|Employee_5141|  Marketing|
|    103|     2|   Employee_2|         HR|
|    103|     7|   Employee_7|         HR|
+-------+--